In [1]:
import pandas as pd
import statsmodels.formula.api as smf
import numpy as np
import os
import gc
# Basic Models with just phase interaction
import sys
import statsmodels.formula.api as smf
from IPython.display import HTML
import numpy as np
import pandas as pd
from stargazer.stargazer import Stargazer

In [2]:
required_columns = [
    'isBounty',
    'userFEIsBounty',
    'userId',
    'timeSinceFirstActivityDays',
    'logtimeSinceFirstActivityDays',
    'userFeLogTimeSinceFirstActivityDays',
    'numHelpProvidedAT',
    'numQuestionsAskedAT',
    'logNumHelpProvidedAT',
    'logNumQuestionsAskedAT',
    'userFeLogNumHelpProvidedAT',
    'userFeLogNumQuestionsAskedAT',
    'initialExperienceReceiving',
    'initialExperienceGiving'
]

df_bounty = pd.read_parquet('user_answers_bounty_processed.parquet', columns=required_columns)

required_columns = [
    'questionFeNumHelped',
    'numHelped',
    'phase',
    'hasAnswer',
    'userId',
    'year',
    'userFeNumHelped',
    'numHelpProvidedAT',
    'numQuestionsAskedAT',
    'logNumHelpProvidedAT',
    'logNumQuestionsAskedAT',
    'timeSinceFirstActivityDays',
    'initialExperienceReceiving',
    'initialExperienceGiving'
]

df_rec = pd.read_parquet('question_centered_model_7d.parquet', columns=required_columns)
df_rec['logtimeSinceFirstActivityDays'] = np.log(1 + df_rec['timeSinceFirstActivityDays'])
print("Column names for df_bounty:")
print(df_bounty.columns.tolist())

print("\nColumn names for df_rec:")
print(df_rec.columns.tolist())

Column names for df_bounty:
['isBounty', 'userFEIsBounty', 'userId', 'timeSinceFirstActivityDays', 'logtimeSinceFirstActivityDays', 'userFeLogTimeSinceFirstActivityDays', 'numHelpProvidedAT', 'numQuestionsAskedAT', 'logNumHelpProvidedAT', 'logNumQuestionsAskedAT', 'userFeLogNumHelpProvidedAT', 'userFeLogNumQuestionsAskedAT', 'initialExperienceReceiving', 'initialExperienceGiving']

Column names for df_rec:
['questionFeNumHelped', 'numHelped', 'phase', 'hasAnswer', 'userId', 'year', 'userFeNumHelped', 'numHelpProvidedAT', 'numQuestionsAskedAT', 'logNumHelpProvidedAT', 'logNumQuestionsAskedAT', 'timeSinceFirstActivityDays', 'initialExperienceReceiving', 'initialExperienceGiving', 'logtimeSinceFirstActivityDays']


## Reciprocity Analysis

In [10]:
filtered_df = df_rec[(df_rec['numQuestionsAskedAT'].isin([0, 1]))]
user_ids_both_0_and_1 = filtered_df.groupby('userId')['numQuestionsAskedAT'].apply(lambda x: set(x) == {0, 1}).reset_index()
user_ids_both_0_and_1 = user_ids_both_0_and_1[user_ids_both_0_and_1['numQuestionsAskedAT'] == True]['userId']
filtered_df = filtered_df[filtered_df['userId'].isin(user_ids_both_0_and_1)]

mean_numHelped_by_user = filtered_df.groupby(['userId', 'phase'])['numHelped'].transform('mean')
filtered_df['numHelpedUserFE'] = filtered_df['numHelped'] - mean_numHelped_by_user

formula = 'numHelpedUserFE ~ phase*C(initialExperienceReceiving)*hasAnswer + phase*hasAnswer*logtimeSinceFirstActivityDays'

# Fit the logistic model using the formula and clustered standard errors
model = smf.ols(formula=formula, data=filtered_df).fit(cov_type='cluster', cov_kwds={'groups': filtered_df['userId']})
print(model.summary())

formula = 'numHelpedUserFE ~ phase*C(initialExperienceReceiving)*hasAnswer + phase*hasAnswer*logtimeSinceFirstActivityDays + phase*hasAnswer*logNumHelpProvidedAT'

# Fit the logistic model using the formula and clustered standard errors
model = smf.ols(formula=formula, data=filtered_df).fit(cov_type='cluster', cov_kwds={'groups': filtered_df['userId']})
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:        numHelpedUserFE   R-squared:                       0.000
Model:                            OLS   Adj. R-squared:                  0.000
Method:                 Least Squares   F-statistic:                     111.8
Date:                Thu, 19 Jun 2025   Prob (F-statistic):               0.00
Time:                        08:27:58   Log-Likelihood:            -1.0824e+07
No. Observations:             8598992   AIC:                         2.165e+07
Df Residuals:                 8598976   BIC:                         2.165e+07
Df Model:                          15                                         
Covariance Type:              cluster                                         
                                                                      coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------

In [11]:
filtered_df = df_rec[(df_rec['numQuestionsAskedAT'].isin([0,1,2,3,4,5])) & (df_rec['numHelpProvidedAT'].isin([0, 1]))]
user_ids_both_0_and_1 = filtered_df.groupby('userId')['numHelpProvidedAT'].apply(lambda x: set(x) == {0, 1}).reset_index()
user_ids_both_0_and_1 = user_ids_both_0_and_1[user_ids_both_0_and_1['numHelpProvidedAT'] == True]['userId']
filtered_df = filtered_df[filtered_df['userId'].isin(user_ids_both_0_and_1)]

mean_numHelped_by_user = filtered_df.groupby(['userId', 'phase'])['numHelped'].transform('mean')
filtered_df['numHelpedUserFE'] = filtered_df['numHelped'] - mean_numHelped_by_user

formula = 'numHelpedUserFE ~ phase * C(initialExperienceGiving) * hasAnswer + phase:C(initialExperienceGiving):hasAnswer'

# Fit the logistic model using the formula and clustered standard errors
model = smf.ols(formula=formula, data=filtered_df).fit(cov_type='cluster', cov_kwds={'groups': filtered_df['userId']})
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:        numHelpedUserFE   R-squared:                       0.001
Model:                            OLS   Adj. R-squared:                  0.001
Method:                 Least Squares   F-statistic:                     97.49
Date:                Thu, 19 Jun 2025   Prob (F-statistic):          1.68e-222
Time:                        08:29:47   Log-Likelihood:            -2.1914e+06
No. Observations:             2057330   AIC:                         4.383e+06
Df Residuals:                 2057318   BIC:                         4.383e+06
Df Model:                          11                                         
Covariance Type:              cluster                                         
                                                                      coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------

## Status Motive Analysis

In [5]:
filtered_df = df_bounty[(df_bounty['numQuestionsAskedAT'].isin([0, 1])) & (df_bounty['numHelpProvidedAT'] < 3)]
user_ids_both_0_and_1 = filtered_df.groupby('userId')['numQuestionsAskedAT'].apply(lambda x: set(x) == {0, 1}).reset_index()
user_ids_both_0_and_1 = user_ids_both_0_and_1[user_ids_both_0_and_1['numQuestionsAskedAT'] == True]['userId']
filtered_df = filtered_df[filtered_df['userId'].isin(user_ids_both_0_and_1)]

mean_isBounty_by_user = filtered_df.groupby('userId')['isBounty'].transform('mean')
filtered_df['isBountyUserFE'] = filtered_df['isBounty'] - mean_isBounty_by_user

# Define the logistic regression formula
formula = 'isBountyUserFE ~ initialExperienceReceiving + logtimeSinceFirstActivityDays + logNumQuestionsAskedAT'

# Fit the logistic model using the formula and clustered standard errors
model = smf.ols(formula=formula, data=filtered_df).fit(cov_type='cluster', cov_kwds={'groups': filtered_df['userId']})

# Print the summary of the model
print(model.summary())


                            OLS Regression Results                            
Dep. Variable:         isBountyUserFE   R-squared:                       0.001
Model:                            OLS   Adj. R-squared:                  0.001
Method:                 Least Squares   F-statistic:                     26.02
Date:                Thu, 19 Jun 2025   Prob (F-statistic):           1.37e-21
Time:                        07:45:57   Log-Likelihood:             3.0160e+05
No. Observations:              236798   AIC:                        -6.032e+05
Df Residuals:                  236794   BIC:                        -6.031e+05
Df Model:                           3                                         
Covariance Type:              cluster                                         
                                                   coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------------

In [6]:
filtered_df = df_bounty[(df_bounty['numQuestionsAskedAT'] > 0) & (df_bounty['numHelpProvidedAT'].isin([0, 1]))]
user_ids_both_0_and_1 = filtered_df.groupby('userId')['numHelpProvidedAT'].apply(lambda x: set(x) == {0, 1}).reset_index()
user_ids_both_0_and_1 = user_ids_both_0_and_1[user_ids_both_0_and_1['numHelpProvidedAT'] == True]['userId']
filtered_df = filtered_df[filtered_df['userId'].isin(user_ids_both_0_and_1)]

mean_isBounty_by_user = filtered_df.groupby('userId')['isBounty'].transform('mean')
filtered_df['isBountyUserFE'] = filtered_df['isBounty'] - mean_isBounty_by_user

# Define the logistic regression formula
formula = 'isBountyUserFE ~ initialExperienceGiving + logtimeSinceFirstActivityDays + logNumQuestionsAskedAT'

# Fit the logistic model using the formula and clustered standard errors
model = smf.ols(formula=formula, data=filtered_df).fit(cov_type='cluster', cov_kwds={'groups': filtered_df['userId']})

# Print the summary of the model
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:         isBountyUserFE   R-squared:                       0.000
Model:                            OLS   Adj. R-squared:                  0.000
Method:                 Least Squares   F-statistic:                     12.96
Date:                Thu, 19 Jun 2025   Prob (F-statistic):           1.50e-10
Time:                        07:46:12   Log-Likelihood:             1.4184e+06
No. Observations:             1047862   AIC:                        -2.837e+06
Df Residuals:                 1047857   BIC:                        -2.837e+06
Df Model:                           4                                         
Covariance Type:              cluster                                         
                                                   coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------------